# 1. Summary Chunk Creation
### Input: Summary.docx
### Output: output/docx_sections/{FocusGroup}.docx

In [1]:
pip install python-docx

Note: you may need to restart the kernel to use updated packages.


In [2]:
from docx import Document
import os

input_path = '../data/Summary.docx'

In [3]:
def split_docx_by_heading(input_path, output_dir):
    import re

    if not os.path.exists(input_path):
        raise FileNotFoundError(f"The file {input_path} does not exist.")

    # Load the document
    doc = Document(input_path)

    # Helper: copy paragraph with runs and basic formatting
    def copy_paragraph(src_para, dst_doc):
        new_para = dst_doc.add_paragraph()
        # Preserve paragraph style by name when possible
        try:
            if src_para.style is not None:
                new_para.style = src_para.style.name
        except Exception:
            pass
        for run in src_para.runs:
            new_run = new_para.add_run(run.text)
            try:
                new_run.bold = run.bold
            except Exception:
                pass
            try:
                new_run.italic = run.italic
            except Exception:
                pass
            try:
                new_run.underline = run.underline
            except Exception:
                pass
            # Copy simple font attributes when available
            try:
                if run.font is not None:
                    if run.font.name:
                        new_run.font.name = run.font.name
                    if run.font.size:
                        new_run.font.size = run.font.size
                    if run.font.color and hasattr(run.font.color, 'rgb') and run.font.color.rgb is not None:
                        new_run.font.color.rgb = run.font.color.rgb
            except Exception:
                pass
            # Preserve character style if named
            try:
                if run.style is not None:
                    new_run.style = run.style.name
            except Exception:
                pass
        return new_para

    # Create chunks of the document based on Heading 2
    sections = []
    current_section = None

    for para in doc.paragraphs:
        style_name = None
        try:
            if para.style is not None:
                style_name = para.style.name
        except Exception:
            style_name = None

        if style_name and style_name.startswith('Heading 2'):
            if current_section is not None:
                sections.append(current_section)
            # store the heading paragraph object and an empty list for its content
            current_section = (para, [])
        elif current_section is not None:
            current_section[1].append(para)

    # Add the last section if it exists
    if current_section is not None:
        sections.append(current_section)

    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    # Save each section as a separate .docx file, preserving styles
    for index, (heading_para, content_paras) in enumerate(sections, start=1):
        section_doc = Document()
        # Copy the heading paragraph (preserves style and runs)
        copy_paragraph(heading_para, section_doc)
        for paragraph in content_paras:
            copy_paragraph(paragraph, section_doc)

        # Use heading text to produce a filename, sanitize for Windows
        heading_text = heading_para.text if heading_para is not None else f'section_{index}'
        clean_heading = re.sub(r'(?i)\bdiscovery\s+session\b', '', heading_text)
        safe_heading = ''.join(ch for ch in clean_heading if ch not in '<>:"/\\|?*').strip()
        safe_heading = re.sub(r'\s{2,}', ' ', safe_heading).strip(' .-_')

        if not safe_heading:
            safe_heading = f'section_{index}'

        output_file_path = os.path.join(output_dir, f'{safe_heading}.docx')
        section_doc.save(output_file_path)
        print(f'Saved: {output_file_path}')

In [4]:
split_docx_by_heading(input_path, '../output/docx_sections')

Saved: ../output/docx_sections/DOCE Office Manager.docx
Saved: ../output/docx_sections/DOCE Admin Aide.docx
Saved: ../output/docx_sections/DOCE Building Inspectors.docx
Saved: ../output/docx_sections/DOCE Housing Inspectors.docx
Saved: ../output/docx_sections/DOCE CommercialPermitElectrical Inspectors.docx
Saved: ../output/docx_sections/DOCE Supervisors.docx
Saved: ../output/docx_sections/DOCE Fire Prevention Bureau.docx
Saved: ../output/docx_sections/DOCE Zoning.docx
Saved: ../output/docx_sections/CPO CPO Co-Ordinator.docx
Saved: ../output/docx_sections/CPO Central Permit Office.docx
Saved: ../output/docx_sections/BAA BAA Supervisors.docx
Saved: ../output/docx_sections/BAA BAA ALJs.docx
Saved: ../output/docx_sections/BAA BAA Ops.docx
Saved: ../output/docx_sections/LAW Law.docx
Saved: ../output/docx_sections/Assessment Assessment Supervisors.docx
Saved: ../output/docx_sections/NBD NBD Internal.docx
Saved: ../output/docx_sections/NBD NBD Data Team.docx
Saved: ../output/docx_sections/CPC